# Prajna-2B Phase 1 — Colab T4 Training (resumable)

Trains the CRN adapter (ReflectiveLoop + resonance + skills + memory) on top of a frozen
`google/gemma-4-E2B` base, in **3 stages**: SFT -> DPO -> Contrastive. Seeds from your proven
`dpo_final.pt` and writes `sft_v2_*` / `dpo_v2_*` / `con_v2_*` to **Google Drive**, so it is
**fully resumable across Colab sessions / disconnects**.

### Before you run
1. Runtime -> Change runtime type -> **T4 GPU**, High-RAM.
2. **If the GitHub repo is private, set `GITHUB_TOKEN` in the Config cell** (clone will fail otherwise).
3. (Optional) Put your own training data on Drive: `MyDrive/prajna_v2/data/error_correction_pairs.json`
   (360 pairs, each `{"prompt","chosen","rejected","domain"}`). If absent, training will ask you to upload it.
4. The CEHRI exam (`cehri_exam.json`, 60 questions G.001-G.060) is **shipped in the repo** and copied
   to Drive automatically. To override it, place your own at `MyDrive/prajna_v2/cehri_exam.json`.
5. The seed `dpo_final.pt` is pulled automatically from the git repo; the base model
   `google/gemma-4-E2B` is downloaded from HF (set `HF_TOKEN` below if it is private).

### Running flow
Run all cells in order. The **Launch training** cell blocks until SFT/DPO/CON finish, then
the eval cells run automatically. To **resume** after a session disconnect:
re-run from the Launch training cell — it loads the latest checkpoint from Drive and continues.

In [ ]:
# ===================== CONFIG =====================
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"   # only if google/gemma-4-E2B is private
GITHUB_TOKEN = ""                       # REQUIRED if the prajna repo is private

DRIVE_ROOT = "/content/drive/MyDrive/prajna_v2"
DRIVE_CKPT = f"{DRIVE_ROOT}/checkpoints"
DRIVE_DATA = f"{DRIVE_ROOT}/data"

SFT_STEPS  = 2000
DPO_STEPS  = 500
CON_STEPS  = 200
BATCH      = 1     # the script hardcodes 1; safe on T4
MAXLEN     = 96    # script reads env CRN_MAXLEN
SAVE_EVERY = 50    # checkpoint + resume granularity (steps)
print("Config ready.")

In [ ]:
# Install deps (Colab ships torch; ensure a recent transformers for gemma-4 tokenizer)
!pip install -q --upgrade transformers accelerate
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, shutil
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)
print("Drive mounted at", DRIVE_ROOT)

In [ ]:
if HF_TOKEN and HF_TOKEN != "PASTE_YOUR_HF_TOKEN_HERE":
    !huggingface-cli login --token {HF_TOKEN}
else:
    print("No HF_TOKEN set -> if model load fails, run:  !huggingface-cli login")

In [ ]:
import os, shutil
repo_dir = "/content/prajna"
if os.path.exists(repo_dir) and not os.path.exists(os.path.join(repo_dir, ".git")):
    print("Removing stale (non-git) repo dir:", repo_dir)
    shutil.rmtree(repo_dir)
if os.path.exists(os.path.join(repo_dir, ".git")):
    print("Repo exists -> git pull")
    !cd {repo_dir} && git pull
else:
    url = "https://github.com/eulogik/prajna.git"
    if GITHUB_TOKEN:
        url = f"https://{GITHUB_TOKEN}@github.com/eulogik/prajna.git"
    print("Cloning", (url.replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else url))
    !git clone {url} {repo_dir}
script = os.path.join(repo_dir, "prajna-phase2/src/train_prajna2b.py")
if not os.path.exists(script):
    raise SystemExit("CLONE FAILED: training script not found at " + script +
                     ".\nIf the repo is private, set GITHUB_TOKEN in the Config cell and re-run this cell.")

In [ ]:
import os, shutil
repo_ckpt = "/content/prajna/prajna/checkpoints"
repo_data = "/content/prajna/prajna/data"

# copy the seed (dpo_final.pt + memory) from the cloned repo into Drive, once
for f in ["dpo_final.pt", "memory_dpo_final.json", "memory_sft_final.json"]:
    src, dst = os.path.join(repo_ckpt, f), os.path.join(DRIVE_CKPT, f)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst); print("copied seed", f)

# copy any data files shipped in the repo (e.g. cehri_exam.json) into Drive, once
if os.path.isdir(repo_data):
    for f in os.listdir(repo_data):
        s, d = os.path.join(repo_data, f), os.path.join(DRIVE_DATA, f)
        if os.path.isfile(s) and not os.path.exists(d):
            shutil.copy(s, d); print("copied data file", f)
else:
    print("repo data dir missing (clone may have failed); skipping data copy")

# replace the repo's local dirs with symlinks -> Drive (so checkpoints/state persist)
for repo_d, drive_d in [(repo_ckpt, DRIVE_CKPT), (repo_data, DRIVE_DATA)]:
    if os.path.islink(repo_d): os.unlink(repo_d)
    elif os.path.exists(repo_d): shutil.rmtree(repo_d)
    os.symlink(drive_d, repo_d)
    print("symlinked", repo_d, "->", drive_d)

seed = os.path.join(DRIVE_CKPT, "dpo_final.pt")
ec   = os.path.join(DRIVE_DATA, "error_correction_pairs.json")
print("seed present:", os.path.exists(seed))
if not os.path.exists(ec):
    print("\n!!! MISSING", ec)
    print("    Upload error_correction_pairs.json to that Drive folder, then re-run.")
    print("    Training will not start until this file is present.")
else:
    print("EC pairs:", len(json.load(open(ec))))
print("exam present:", os.path.exists(os.path.join(DRIVE_DATA, "cehri_exam.json")))

In [ ]:
import os
os.chdir("/content/prajna")
os.environ.update({
    "CRN_DEVICE": "cuda",
    "CRN_MAXLEN": str(MAXLEN),
    "SFT_V2_STEPS": str(SFT_STEPS),
    "DPO_V2_STEPS": str(DPO_STEPS),
    "CON_V2_STEPS": str(CON_STEPS),
    "SAVE_EVERY": str(SAVE_EVERY),
})
log_path = f"{DRIVE_ROOT}/train.log"
print("Training started (blocking — this cell runs until all 3 stages finish).")
print("Log ->", log_path)
!python3 -u prajna-phase2/src/train_prajna2b.py 2>&1 | tee {log_path}
print("Training complete.")

In [ ]:
# ============ Evaluation: error-correction rate ============
import os, sys, json, random, torch
os.chdir("/content/prajna")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
sys.path.insert(0, "prajna-phase2/src")
from crn_components import PrajnaStudentMultiLayer

CKPT = f"{DRIVE_CKPT}/dpo_v2_final.pt"
MEM  = f"{DRIVE_CKPT}/memory_v2_final.json"
if not os.path.exists(CKPT):
    print("No final checkpoint at", CKPT, " — training may not have finished yet.")
    print("Run the Launch Training cell first and wait for it to complete.")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    student = PrajnaStudentMultiLayer(device=device, inject_every=8, max_length=96, crn_mix_init=0.05)
    student.to(device)
    sd = torch.load(CKPT, map_location=device, weights_only=False)
    student.load_state_dict(sd['crn'], strict=False)
    if os.path.exists(MEM): student.load_memory(MEM)
    student.eval(); tok = student.tok
    print("reflection_gate(sigmoid):", [f"{x:.3f}" for x in torch.sigmoid(student.reflection_gate).tolist()])

    @torch.no_grad()
    def gen_crn(prompt, max_new=96):
        ids = tok(prompt, return_tensors='pt').input_ids.to(device)
        g = ids.clone()
        for _ in range(max_new):
            o = student._collect_hidden(g)
            lg, _ = student._apply_crn(o, training=False)
            nt = lg[:, -1, :].argmax(-1).reshape(1, 1)
            g = torch.cat([g, nt], dim=1)
            if nt.item() == tok.eos_token_id: break
        return tok.decode(g[0], skip_special_tokens=True)[len(prompt):].strip()

    @torch.no_grad()
    def gen_base(prompt, max_new=96):
        ids = tok(prompt, return_tensors='pt').input_ids.to(device)
        out = student.base_model.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

    def correct(out, gold): return gold.strip().lower() in out.strip().lower()

    ec = json.load(open(f"{DRIVE_DATA}/error_correction_pairs.json"))
    random.seed(7); random.shuffle(ec)
    held = [e for e in ec if e['domain'] in ('math', 'facts', 'igr')][:100]
    base_wrong = [e for e in held if not correct(gen_base(e['prompt']), e['chosen'])]
    fixed = sum(correct(gen_crn(e['prompt']), e['chosen']) for e in base_wrong)
    rate = fixed / len(base_wrong) if base_wrong else 0
    print(f"\n[ERROR-CORRECTION RATE] CRN fixes {fixed}/{len(base_wrong)} = {rate*100:.1f}% of base errors")

In [ ]:
# ============ Evaluation: CEHRI licensing exam (G.001-G.060) ============
import os, json, torch
exam_path = f"{DRIVE_ROOT}/cehri_exam.json"
gen_crn = globals().get('gen_crn'); student = globals().get('student'); tok = globals().get('tok')
if not (gen_crn and student and tok):
    print("Run the Evaluation cell above first (it loads the model and defines gen_crn).")
elif not os.path.exists(exam_path):
    print("No cehri_exam.json at", exam_path, "(it is normally copied from the repo automatically).")
else:
    exam = json.load(open(exam_path))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    passed = 0
    for q in exam:
        out = gen_crn(q['prompt'], max_new=120)
        ok = q['answer'].strip().lower() in out.strip().lower()
        passed += ok
        print(f"  {q['id']}: {'PASS' if ok else 'FAIL'}  {out[:70]!r}")
    frac = passed / len(exam)
    print(f"\nCEHRI RESULT: {passed}/{len(exam)} = {frac*100:.1f}%  -> "
          f"{'PASS' if frac >= 0.9 else 'FAIL (<0.9)'}")